In [1]:
import pandas as pd
import numpy as np
import glob
import os
import sys
sys.path.insert(0, '../')
#sys.path.insert(0, './')
from data.Reinhard import Reinhard
from models.model_mrcnn import _default_mrcnn_config, build_default
from visualization.explain import ExplainPredictions
import torch
import tqdm
from PIL import Image
import torchvision
from torchmetrics.classification import MulticlassConfusionMatrix
import torchvision.ops.boxes as bops
from torchmetrics.classification import Dice
from PIL import Image
from skimage import draw
from skimage import measure
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np
import random

In [2]:
dice = Dice(average='samples')
preds = torch.tensor([2, 0, 2, 1])
target = torch.tensor([1, 1, 2, 0])
dice(preds, target)

tensor(0.2500)

In [3]:
#LBD_model_path="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/models/mrcnn_models/effortless-violet-306_mrcnn_model_74.pth" # new model
LBD_model_path="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/models/mrcnn_models/wild-waterfall-307_mrcnn_model_24.pth" # new model 2

In [4]:
test_config = dict(batch_size = 1, num_classes = 3)
model_config = _default_mrcnn_config(num_classes=1 + test_config['num_classes']).config
model_lbd = build_default(model_config, im_size=1024)

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
model_lbd.load_state_dict(torch.load(LBD_model_path))

<All keys matched successfully>

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_lbd.eval().to(device)

GeneralizedRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.8224, 0.7101, 0.7279], std=[0.094, 0.2354, 0.2226])
      Resize(min_size=(1024,), max_size=1024, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
    

In [7]:
#output_folder = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/end2end_model_preds/effortless-violet-306_mrcnn_model_74.pth_3uctrn1o10.pth_stride_256"
output_folder = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/end2end_model_preds/wild-waterfall-307_mrcnn_model_24.pth_3uctrn1o10.pth"

In [9]:
davids_annot_folder = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/end2end_model_preds/woven-wood-253_mrcnn_model_24.pth_3uctrn1o10.pth_test_data2"


In [30]:
imgs = ["14_133_FCx_aSyn_x200.svs","16_044_PCx_aSyn_x200.svs","PD268_Syn1_CG.svs","PD110_Syn1_TCx.svs"]

In [31]:
def FindPoint(x1, y1, x2, 
              y2, x, y) :
    if (x > x1 and x < x2 and
        y > y1 and y < y2) :
        return True
    else :
        return False
    
def find_intersection(david_coords, row):
    #x1,y1,x2,y2 =  row[5]-row[3], row[6]-row[1],row[5]+row[4],row[6]+row[2] 
    x1,y1,x2,y2 =  row["img_x"]-row["box_y1"], row["img_y"]-row["box_x1"],row["img_x"]+row["box_y2"],row["img_y"]+row["box_x2"] 
    for x,y in david_coords.sort_values(['x','y']).values:
        f = FindPoint(x1, y1, x2,  y2, x, y)
        if f:
            return True
    return False

def show_images(images, cols = 1, titles = None):
    """Display a list of images in a single figure with matplotlib.
    
    Parameters
    ---------
    images: List of np.arrays compatible with plt.imshow.
    
    cols (Default = 1): Number of columns in figure (number of rows is 
                        set to np.ceil(n_images/float(cols))).
    
    titles: List of titles corresponding to each image. Must have
            the same length as titles.
    """
    assert((titles is None)or (len(images) == len(titles)))
    n_images = len(images)
    if titles is None: titles = ['Image (%d)' % i for i in range(1,n_images + 1)]
    fig = plt.figure()
    for n, (image, title) in enumerate(zip(images, titles)):
        a = fig.add_subplot(cols, int(np.ceil(n_images/float(cols))), n + 1)
        if image.ndim == 2:
            plt.gray()
        plt.imshow(image)
        #a.set_title(title)
    fig.set_size_inches(np.array(fig.get_size_inches()) * n_images)
    print(fig.get_size_inches())
    plt.show()
    

def Rand(start, end, num=4):
    res = []
    for j in range(num):
        res.append(np.random.randint(start, end))
 
    return res
 
 
def find_matched_point(coords, row):
    #coords1 = coords["image_name", "label",	"confidence", "box_x1", "box_x2", "box_y1", "box_y2", "img_x", "img_y"]
    coords["x1"] = coords["img_x"]-coords["box_y1"]
    coords["y1"] = coords["img_y"]-coords["box_x1"]
    coords["x2"] = coords["img_x"]+coords["box_y2"]
    coords["y2"] = coords["img_y"]+coords["box_x2"]
    x = row["x"]
    y = row["y"]
    for index, r in coords.iterrows():
        f = FindPoint(r["x1"], r["y1"], r["x2"], r["y2"], x, y)
        if f:
            return index
    return None


In [36]:
#img_name = "16_044_PCx_aSyn_x200.svs"
def find_missed_points(img_name):
    coords = pd.read_csv(os.path.join(output_folder, img_name, img_name.replace(".svs",".csv")))
    coords.sort_values(['img_x','img_y'])
    david_coords = pd.read_csv(os.path.join(davids_annot_folder,img_name,img_name+"-points.csv"))
    david_coords.sort_values(['x','y']).values
    coords["match"] = coords.apply(lambda l: find_intersection(david_coords,l), axis=1)
    print("*********************", img_name,"************************")
    print(len(coords))
    print(len(david_coords))
    print("matched "+img_name+": ",  coords["match"].sum())
    print("matched with 0.65 conf "+img_name+":",len(coords[(coords["match"]==True) & (coords["confidence"]>=0.65)]))
    print("matched true/pre "+img_name+": ", coords[coords["False"]!=1]["match"].sum())

    david_coords_pd = pd.DataFrame(david_coords.sort_values(['x','y']).values, columns=["x","y"])
    david_coords_pd["found_match"] = david_coords_pd.apply(lambda row: find_matched_point(coords, row),axis=1)

    model_missed = david_coords_pd[david_coords_pd["found_match"].isna()]
    david_missed = coords[coords["match"].isna()]
    return model_missed, david_missed

In [37]:
imgs = coords[(coords["match"]==False) & (coords["confidence"]>=0.5) & (coords["True"]==1)]["image_name"].values
images = [np.asarray(Image.open(os.path.join(output_folder, img_name)+"/detections/"+img+"_detection.png")) for img in imgs]
start = 0
end = len(images)
img_to_pick = Rand(start, end, num=4)
show_images(img_to_pick, cols =1, titles = None)

KeyError: 'match'

In [38]:
for img_name in imgs:
    find_missed_points(img_name)

********************* 14_133_FCx_aSyn_x200.svs ************************
107
133
matched 14_133_FCx_aSyn_x200.svs:  51
matched with 0.65 conf 14_133_FCx_aSyn_x200.svs: 39
matched true/pre 14_133_FCx_aSyn_x200.svs:  40
********************* 16_044_PCx_aSyn_x200.svs ************************
309
123
matched 16_044_PCx_aSyn_x200.svs:  72
matched with 0.65 conf 16_044_PCx_aSyn_x200.svs: 71
matched true/pre 16_044_PCx_aSyn_x200.svs:  71
********************* PD268_Syn1_CG.svs ************************
47
151
matched PD268_Syn1_CG.svs:  34
matched with 0.65 conf PD268_Syn1_CG.svs: 33
matched true/pre PD268_Syn1_CG.svs:  34
********************* PD110_Syn1_TCx.svs ************************
70
115
matched PD110_Syn1_TCx.svs:  36
matched with 0.65 conf PD110_Syn1_TCx.svs: 30
matched true/pre PD110_Syn1_TCx.svs:  36


,Unnamed: 0,image_name,label,confidence,brown_pixels,True,Pre,False,centroid,eccentricity,area,equivalent_diameter,box_x1,box_x2,box_y1,box_y2,img_x,img_y,x,y
0,0,16_044_PCx_aSyn_x200_3584_29696,Pre,0.997662,0,0,1,0,"(8.866840731070496, 12.109660574412533)",0.661633,383,22.082816,666,684,413,437,3584,29696,4009.0,30371.0
1,1,16_044_PCx_aSyn_x200_3584_29696,Pre,0.753374,0,0,1,0,"(9.13986013986014, 12.237762237762238)",0.464426,429,23.371345,875,898,508,532,3584,29696,4104.0,30582.5
2,0,16_044_PCx_aSyn_x200_3584_30720,Pre,0.985457,0,0,1,0,"(10.404109589041095, 9.876712328767123)",0.484063,438,23.615226,714,736,855,876,3584,30720,4449.5,31445.0
3,1,16_044_PCx_aSyn_x200_3584_30720,Pre,0.969375,0,0,1,0,"(16.421259842519685, 13.23031496062992)",0.739339,508,25.432375,919,949,816,843,3584,30720,4413.5,31654.0
4,0,16_044_PCx_aSyn_x200_3584_31744,Pre,0.991198,0,0,1,0,"(15.262608695652174, 11.140869565217391)",0.754429,575,27.057582,830,862,747,772,3584,31744,4343.5,32590.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304,1,16_044_PCx_aSyn_x200_42496_24576,Pre,0.946276,0,0,1,0,"(13.241496598639456, 13.695578231292517)",0.825191,588,27.361741,296,320,144,177,42496,24576,42656.5,24884.0
305,0,16_044_PCx_aSyn_x200_42496_25600,Pre,0.993005,0,0,1,0,"(10.944954128440367, 12.910091743119265)",0.664061,545,26.342277,64,87,349,377,42496,25600,42859.0,25675.5
306,1,16_044_PCx_aSyn_x200_42496_25600,Pre,0.864903,0,0,1,0,"(11.212550607287449, 12.18421052631579)",0.639956,494,25.079480,341,363,361,390,42496,25600,42871.5,25952.0
307,0,16_044_PCx_aSyn_x200_42496_27648,Pre,0.990344,0,0,1,0,"(8.822323462414579, 12.801822323462414)",0.629087,439,23.642169,425,444,174,200,42496,27648,42683.0,28082.5
